# DAG Utils Testing

In [1]:
%cd ..

from dag import QuantumDAG

/workspaces/sqa26pro16


In [2]:
# Create a new quantum DAG with 4 qubits
dag = QuantumDAG(num_qubits=4)

# Add initial Hadamard gates (layer 0)
dag.add_gate('H', [0])
dag.add_gate('H', [1])
dag.add_gate('H', [2])
dag.add_gate('H', [3])

# Add CNOT gates (layer 1)
dag.add_gate('CNOT', [0, 1])  # Control: q0, Target: q1
dag.add_gate('CNOT', [2, 3])  # Control: q2, Target: q3

# Add more gates (layer 2)
dag.add_gate('RZ', [1], parameters={'theta': 0.5})
dag.add_gate('RZ', [3], parameters={'theta': 0.3})

# Add a cross-partition CNOT (layer 3) - this will span processors
dag.add_gate('CNOT', [1, 2])  # This crosses the partition boundary

# Add final gates (layer 4)
dag.add_gate('H', [1])
dag.add_gate('H', [2])

print(f"Total gates added: {dag.get_gate_count()}")
print(f"Circuit depth: {dag.get_depth()}")

Total gates added: 11
Circuit depth: 5


In [3]:
# Get the front layer (gates with no dependencies)
front = dag.get_front_layer()
print("Front Layer (depth 0):")
for gate_id in front:
    print(f"  {dag.gates[gate_id]}")

Front Layer (depth 0):
  Gate(g0: H on qubits [0])
  Gate(g1: H on qubits [1])
  Gate(g2: H on qubits [2])
  Gate(g3: H on qubits [3])


In [4]:
# Get the back layer (final gates)
back = dag.get_back_layer()
print("Back Layer (final gates):")
for gate_id in back:
    print(f"  {dag.gates[gate_id]}")

Back Layer (final gates):
  Gate(g7: RZ on qubits [3])
  Gate(g9: H on qubits [1])
  Gate(g10: H on qubits [2])


In [5]:
def print_extension_layers(dag, depth=None):
    print("\nExtension Layers (depth-based):")
    max_depth = dag.get_depth()
    end = max_depth if depth is None else min(depth + 1, max_depth)

    for distance in range(end):
        layer = dag.get_extended_layer(distance)
        gate_info = [f"{dag.gates[g].gate_type}(q{dag.gates[g].qubits})" for g in layer]
        print(f"  Layer {distance}: {gate_info}")

print_extension_layers(dag, 1)


Extension Layers (depth-based):
  Layer 0: []
  Layer 1: ['CNOT(q[0, 1])', 'CNOT(q[2, 3])']


In [6]:
# dag.draw()

# Verifying DAG for Single Core Processor

In [7]:
%pip install qiskit -qqq
%pip install networkx -qqq
%pip install matplotlib -qqq

print("Packages Installed Successfully")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Packages Installed Successfully


In [8]:
from typing import List, Dict, Tuple, Set, Optional
from dataclasses import dataclass
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit import DAGCircuit, DAGOpNode
import networkx as nx

%cd sqa26pro16/

# from src.dag import QuantumDAG
from convert import from_qiskit, from_qasm_file

[Errno 2] No such file or directory: 'sqa26pro16/'
/workspaces/sqa26pro16


In [9]:
@dataclass
class VerificationResult:
    """Result of DAG verification."""
    is_valid: bool
    gate_count_match: bool
    topological_order_valid: bool
    dependencies_match: bool
    errors: List[str]
    warnings: List[str]
    
    def __repr__(self):
        status = "✓ PASSED" if self.is_valid else "✗ FAILED"
        return f"VerificationResult({status}, errors={len(self.errors)})"
    
    def print_report(self):
        """Print detailed verification report."""
        print("=" * 60)
        print("DAG Verification Report")
        print("=" * 60)
        
        status = "✓ PASSED" if self.is_valid else "✗ FAILED"
        print(f"Overall: {status}")
        print(f"  Gate count match: {'✓' if self.gate_count_match else '✗'}")
        print(f"  Topological order valid: {'✓' if self.topological_order_valid else '✗'}")
        print(f"  Dependencies match: {'✓' if self.dependencies_match else '✗'}")
        
        if self.errors:
            print(f"\nErrors ({len(self.errors)}):")
            for err in self.errors:
                print(f"  ✗ {err}")
        
        if self.warnings:
            print(f"\nWarnings ({len(self.warnings)}):")
            for warn in self.warnings:
                print(f"  ⚠ {warn}")
        
        print("=" * 60)

In [10]:
class DAGVerifier:
    """Verifies QuantumDAG against Qiskit's DAGCircuit."""
    
    def __init__(self, quantum_dag, qiskit_circuit: QuantumCircuit):
        """
        Initialize verifier.
        
        Args:
            quantum_dag: Your QuantumDAG instance
            qiskit_circuit: Original Qiskit circuit
        """
        self.qdag = quantum_dag
        self.circuit = qiskit_circuit
        self.qiskit_dag = circuit_to_dag(qiskit_circuit)
        
        self.errors = []
        self.warnings = []
    
    def verify(self) -> VerificationResult:
        """
        Run all verification checks.
        
        Returns:
            VerificationResult with detailed findings
        """
        self.errors = []
        self.warnings = []
        
        gate_count_ok = self._verify_gate_count()
        topo_ok = self._verify_topological_order()
        deps_ok = self._verify_dependencies()
        
        is_valid = gate_count_ok and topo_ok and deps_ok
        
        return VerificationResult(
            is_valid=is_valid,
            gate_count_match=gate_count_ok,
            topological_order_valid=topo_ok,
            dependencies_match=deps_ok,
            errors=self.errors.copy(),
            warnings=self.warnings.copy()
        )
    
    def _get_qiskit_op_nodes(self) -> List[DAGOpNode]:
        """Get operation nodes from Qiskit DAG (excluding barriers)."""
        return [
            node for node in self.qiskit_dag.op_nodes()
            if node.op.name != 'barrier'
        ]
    
    def _verify_gate_count(self) -> bool:
        """Verify gate counts match."""
        qiskit_gates = self._get_qiskit_op_nodes()
        qiskit_count = len(qiskit_gates)
        our_count = self.qdag.get_gate_count()
        
        if qiskit_count != our_count:
            self.errors.append(
                f"Gate count mismatch: Qiskit={qiskit_count}, Ours={our_count}"
            )
            return False
        return True
    
    def _verify_topological_order(self) -> bool:
        """Verify our topological order respects all dependencies."""
        try:
            our_topo = self.qdag.topological_sort()
        except ValueError as e:
            self.errors.append(f"Topological sort failed: {e}")
            return False
        
        # Check that every gate appears exactly once
        if len(our_topo) != len(set(our_topo)):
            self.errors.append("Duplicate gates in topological order")
            return False
        
        if len(our_topo) != self.qdag.get_gate_count():
            self.errors.append("Topological order missing gates")
            return False
        
        # Verify order respects dependencies
        position = {gate_id: i for i, gate_id in enumerate(our_topo)}
        
        for gate_id in our_topo:
            for pred in self.qdag.get_predecessors(gate_id):
                if position[pred] >= position[gate_id]:
                    self.errors.append(
                        f"Invalid order: {pred} should come before {gate_id}"
                    )
                    return False
        
        return True
    
    def _verify_dependencies(self) -> bool:
        """
        Verify dependencies match Qiskit's DAG.
        
        Two gates have a dependency if they share a qubit and one must
        execute before the other. We compare dependency STRUCTURE, not
        exact topological order (since parallel gates can be ordered
        differently while both being valid).
        """
        # Build gate signature -> list of gates mapping
        # Signature = (gate_type, qubits_tuple)
        
        our_gates_by_sig = {}
        for gate_id in self.qdag.topological_sort():
            gate = self.qdag.gates[gate_id]
            sig = (gate.gate_type.upper(), tuple(gate.qubits))
            if sig not in our_gates_by_sig:
                our_gates_by_sig[sig] = []
            our_gates_by_sig[sig].append(gate_id)
        
        qiskit_gates_by_sig = {}
        for node in self.qiskit_dag.topological_op_nodes():
            if node.op.name == 'barrier':
                continue
            qubits = tuple(self.circuit.find_bit(q).index for q in node.qargs)
            sig = (node.op.name.upper(), qubits)
            if sig not in qiskit_gates_by_sig:
                qiskit_gates_by_sig[sig] = []
            qiskit_gates_by_sig[sig].append(node)
        
        # Check same set of gate signatures
        if set(our_gates_by_sig.keys()) != set(qiskit_gates_by_sig.keys()):
            our_only = set(our_gates_by_sig.keys()) - set(qiskit_gates_by_sig.keys())
            qiskit_only = set(qiskit_gates_by_sig.keys()) - set(our_gates_by_sig.keys())
            if our_only:
                self.errors.append(f"Gates only in our DAG: {our_only}")
            if qiskit_only:
                self.errors.append(f"Gates only in Qiskit DAG: {qiskit_only}")
            return False
        
        # Check same counts per signature
        for sig in our_gates_by_sig:
            if len(our_gates_by_sig[sig]) != len(qiskit_gates_by_sig[sig]):
                self.errors.append(
                    f"Gate count mismatch for {sig}: "
                    f"Ours={len(our_gates_by_sig[sig])}, "
                    f"Qiskit={len(qiskit_gates_by_sig[sig])}"
                )
                return False
        
        # Verify per-qubit gate ordering (the key structural check)
        return self._verify_qubit_orderings_structural()
    
    def _verify_qubit_orderings_structural(self) -> bool:
        """
        Verify that gates on each qubit appear in the same relative order.
        
        This is the key structural check - for each qubit, extract the sequence
        of gates that touch it and verify they match between our DAG and Qiskit's.
        """
        
        # Get per-qubit gate sequences from our DAG
        our_qubit_sequences = {q: [] for q in range(self.qdag.num_qubits)}
        for gate_id in self.qdag.topological_sort():
            gate = self.qdag.gates[gate_id]
            sig = (gate.gate_type.upper(), tuple(gate.qubits))
            for q in gate.qubits:
                our_qubit_sequences[q].append(sig)
        
        # Get per-qubit gate sequences from Qiskit DAG
        qiskit_qubit_sequences = {q: [] for q in range(self.circuit.num_qubits)}
        for node in self.qiskit_dag.topological_op_nodes():
            if node.op.name == 'barrier':
                continue
            qubits = tuple(self.circuit.find_bit(q).index for q in node.qargs)
            sig = (node.op.name.upper(), qubits)
            for q in qubits:
                qiskit_qubit_sequences[q].append(sig)
        
        # Compare sequences
        all_ok = True
        for q in range(self.qdag.num_qubits):
            our_seq = our_qubit_sequences.get(q, [])
            qiskit_seq = qiskit_qubit_sequences.get(q, [])
            
            if our_seq != qiskit_seq:
                self.errors.append(
                    f"Gate sequence mismatch on qubit {q}:\n"
                    f"    Ours:   {our_seq}\n"
                    f"    Qiskit: {qiskit_seq}"
                )
                all_ok = False
        
        return all_ok
    
    def _verify_qubit_orderings(self, our_gates: List, qiskit_gates: List) -> bool:
        """Legacy method - kept for compatibility."""
        return self._verify_qubit_orderings_structural()
    
    def compare_detailed(self) -> Dict:
        """
        Get detailed comparison data for debugging.
        
        Returns:
            Dictionary with comparison details
        """
        qiskit_ops = self._get_qiskit_op_nodes()
        
        our_gates = []
        for gate_id in self.qdag.topological_sort():
            gate = self.qdag.gates[gate_id]
            predecessors = self.qdag.get_predecessors(gate_id)
            successors = self.qdag.get_successors(gate_id)
            our_gates.append({
                'id': gate_id,
                'type': gate.gate_type,
                'qubits': gate.qubits,
                'layer': gate.layer,
                'predecessors': predecessors,
                'successors': successors
            })
        
        qiskit_gates = []
        for node in self.qiskit_dag.topological_op_nodes():
            if node.op.name == 'barrier':
                continue
            qubits = [self.circuit.find_bit(q).index for q in node.qargs]
            preds = [
                n.op.name for n in self.qiskit_dag.predecessors(node)
                if isinstance(n, DAGOpNode)
            ]
            succs = [
                n.op.name for n in self.qiskit_dag.successors(node)
                if isinstance(n, DAGOpNode)
            ]
            qiskit_gates.append({
                'type': node.op.name,
                'qubits': qubits,
                'predecessors': preds,
                'successors': succs
            })
        
        return {
            'our_gate_count': len(our_gates),
            'qiskit_gate_count': len(qiskit_gates),
            'our_depth': self.qdag.get_depth(),
            'qiskit_depth': self.qiskit_dag.depth(),
            'our_gates': our_gates,
            'qiskit_gates': qiskit_gates
        }

In [11]:
def verify_dag(qiskit_circuit: QuantumCircuit) -> VerificationResult:
    """
    Convenience function to verify a QuantumDAG against a Qiskit circuit.
    
    Args: 
        qiskit_circuit: Original Qiskit circuit
        
    Returns:
        VerificationResult
    """
    # Convert to our DAG
    our_dag = from_qiskit(qiskit_circuit)

    verifier = DAGVerifier(our_dag, qiskit_circuit)

    return verifier.verify()

In [12]:
def quick_verify(qiskit_circuit: QuantumCircuit) -> bool:
    """
    Quick verification returning True/False.
    
    Usage:
        from dag_utils import QuantumDAG
        
        qc = QuantumCircuit(2)
        qc.h(0)
        qc.cx(0, 1)
        
        if quick_verify(qc, QuantumDAG):
            print("DAG implementation is correct!")
    """
    try:
        our_dag = from_qiskit(qiskit_circuit)
        result = verify_dag(our_dag, qiskit_circuit)
        return result.is_valid
    except Exception:
        return False

In [14]:
%cd workspaces/sqa26pro16/

import qiskit.qasm2

qc = qiskit.qasm2.load("./data/example_4q/example_4q.qasm")

qc.draw(fold=-1)

[Errno 2] No such file or directory: 'workspaces/sqa26pro16/'
/workspaces/sqa26pro16


┌───┐     ┌───┐┌───┐┌───┐┌───┐     
q_0: ──■──┤ Z ├──■──┤ Y ├┤ Y ├┤ Y ├┤ Y ├─────
     ┌─┴─┐└───┘  │  └───┘├───┤├───┤├───┤┌───┐
q_1: ┤ X ├──■────┼────■──┤ X ├┤ X ├┤ X ├┤ X ├
     └───┘  │    │  ┌─┴─┐└───┘└───┘└───┘└───┘
q_2: ───────┼────┼──┤ X ├────────────────────
          ┌─┴─┐┌─┴─┐└───┘                    
q_3: ─────┤ X ├┤ X ├─────────────────────────
          └───┘└───┘

In [15]:
verify_dag(qc)

VerificationResult(✓ PASSED, errors=0)